***

# Question 1: Clinic Queue Simulation

## Supplementary Jupyter Notebook for Mathematical Modelling for Sustainable Development Coursework

***

Methods and assumptions are explained/modelled throughout. Inline comments are left to show how I coded this logically and not meant to be explanative - the markdown blocks serve this purpose!


Full development of this code can be found in the following link: 
[Link to Github Repo](https://github.com/Leonie-G-B/MathModSusDev)


> Student Num. 2101377

> Email: ch21886@bristol.ac.uk

***

### Nomenclature and utility functions: 

Getting this out the way to reduce code clutter later on

In [ ]:
##### Standard inputs & type hinting helpers
import numpy as np
import pandas as pd
from typing import Literal, TypedDict, Sequence
from enum import StrEnum

# Simulation specifics
from collections import deque

# Plots
import seaborn as sbn
import matplotlib.pyplot as plt


##### Type hinting classes
class ServiceMethods(StrEnum): 
    EXPONENTIAL = "exponential"
    LOGNORMAL = "lognormal"    
    NORMAL = "normal"

class ClinicianConfig(TypedDict): # define a type of shift for n number of clinicians
    shift_pattern : tuple[float, float] # list of (start, end)
    appointment_length : float # in minutes, list must match 

class PerformanceMetrics(StrEnum): 
    AVERAGE_WAIT = "avg_wait"
    PEAKS = "peak"
    EOD_PATIENTS = "eod_patients"
    PATIENTS_SERVED = "patients_served"
    WORKLOAD = "avg_clinician_workload"


##### Utility functions 

def format_time_hours(val: float) -> str:
    "Convert decimal hr numbers into something human readable and clear."
    if val >= 1.0:
        hours = int(val)
        minutes = int(round((val - hours) * 60))
        return f"{hours}h {minutes}m" if minutes > 0 else f"{hours}h"
    else:
        minutes = int(round(val * 60))
        return f"{minutes}mins"
    
def build_df_for_plot(
    results_dicts: Sequence[dict[float, dict]],
    labels: Sequence[str],
    sweep_metric: tuple[str, Sequence[float]],
    metric: PerformanceMetrics = "avg_wait",
    calc: str = "mean"
    ) -> pd.DataFrame:

    param_name, values = sweep_metric
    rows: list[dict[str, float | str]] = []
    for res, label in zip(results_dicts, labels):
        for val in sorted(values):
            rows.append({
                param_name: val,
                "value": res[val]["metrics"][metric][calc],
                "std": res[val]["metrics"][metric]["std"],
                "label": label
            })

    return pd.DataFrame(rows)

def print_sweep_results_quick(
        results, sweep_metric: tuple[str, list], 
        sim_inputs: dict, 
        metric: PerformanceMetrics = "avg_wait", calc: str = "mean",
        convert_to_readable_time: bool = False,
        baseline_results: dict | None = None): # if provided, numbers will be quoted as a delta to baseline

    print(f"\nSweep results: {metric} for swept vals of {sweep_metric[0]}")
    # print(f"Constant sim metrics: \n{sim_inputs}")
    x = []
    y = []
    for item in results:
        y_val = results[item]["metrics"][metric][calc]

        pct_str = ""
        if baseline_results is not None:
            appt = item.split("_")[0]
            base_key = f"{appt}_baseline"
            
            if base_key in baseline_results:
                base_val = baseline_results[base_key]["metrics"][metric][calc]

                if base_val != 0:
                    pct_change = (y_val - base_val) / base_val * 100
                    pct_str = f" ({pct_change:+.1f}%)"
                else:
                    pct_str = " (baseline=0)" #so it doesn't break if baseline is 0
            else:
                pct_str = " (no baseline)" 

        if convert_to_readable_time:
            val_str = format_time_hours(y_val)
        else:
            val_str = f"{y_val}"

        print(f"{item} = {val_str}{pct_str}")

        x.append(item)
        y.append(y_val)

    return x, y

***

## Key objects: Clinician and Simulation

### Summary of assumptions and simulation architecture 
* Service times & arrival rates (mu, lambda) are randomly and memoryless. See more info & plots on this on subsequent blocks.
* Single common queue, First Come First Served (FCFS). The size of the 'waiting room' has not been restricted and assumed infinite. 
* The simulation is a fixed duration "day" of the clinic. The time of the start & end of the day (plus the clinicians' work schedules) are required inputs. This was done to allow the investigation into changing the schedules of the clinicians throughout a day of work clearly.


Note: This block is lengthy as it is the complete Clinic Simulation for all subsequent blocks.

***

In [ ]:

class Clinician: 
    def __init__(self, id, appointment_time: int, 
                 shift_start: float,
                 shift_end: float):
        self.id : int = id
        self.mu : float = 60/appointment_time
        
        self.shift_start : float = shift_start
        self.shift_end   : float = shift_end

        self.available : bool = True
        self.next_available: float = None #float time of when they are next free

        self.total_appointment_time : float = 0.0
        self.total_downtime : float = 0.0
        self.last_event_time : float = shift_start #to calculate downtime between appointments
    

    def appointment_start(self, current_time: float): 
        self.total_downtime += current_time - self.last_event_time #assuming last event is finihsing an appointment

        self.available = False
        self.last_event_time = current_time

    def appointment_end(self, current_time: float):
        self.total_appointment_time += current_time - self.last_event_time #we could assume apppointment length but this is more foolproof incase of different EOD behaviour

        self.available = True
        self.last_event_time = current_time
    
    def calc_workload(self):
        shift_length = self.shift_end - self.shift_start
        workload = (shift_length - self.total_downtime) / shift_length
        self.workload = workload


class ClinicSim: 
    def __init__(sim, lambda_base: int, appointment_time: int, 
                 service_method: ServiceMethods = "exponential",
                 peak_multiplier: int = None, #should be 2,4,8 - use checking?
                 open_close: tuple[float, float] = (8.0, 17.5),
                 peak_hrs: tuple[float, float] = (10.0, 14.0), 
                 peak_normal_shape: bool = False, #if True, then creates a normal curve within the peak with max value peak_multiplier. Otherwise uses other default methods.
                 **kwargs):
        
        sim.lambda_base = lambda_base
        sim.lambdas_t = []
        
        sim.peak_multiplier: int = peak_multiplier
        sim.normal_peak: bool = peak_normal_shape
        sim.peak_start = peak_hrs[0]
        sim.peak_end = peak_hrs[1]

        sim.mu = 60/appointment_time #hourly rate

        sim.open_time = open_close[0]
        sim.close_time = open_close[1]

        sim.clock = sim.open_time # start at the start!
        
        sim.queue = deque()
        sim.waits = []# list of the waiting times

        sim.num_in_system = 0
        sim.sys_state = [(sim.clock,0)] #tuples, (time, no. patients in queue) 

        sim.arrival_times = []
        sim.departure_times = [] #just for data logging reasons

        sim.self_set_service_method(service_method, **kwargs) #can pass in "logn_simga" for example for lognormal serv. method

        # sim.servers = [None] * sim.num_clinicians #none indicates free server
        sim.clinicians: list[Clinician] = []

        sim.t_arrival = sim.clock + sim.generate_interarrival()


    def self_set_service_method(sim, method: ServiceMethods, **kwargs): 
        mean_serv_hrs = 1 /sim.mu
        if method == "exponential": 
            sim._service_func = lambda: np.random.exponential(mean_serv_hrs)
        elif method == "lognormal": 
            sigma = kwargs.get("logn_sigma", 0.5)
            mu_log = np.log(mean_serv_hrs) - 0.5 * sigma**2
            sim._service_func = lambda: np.random.lognormal(mean= mu_log, sigma = sigma)
        elif method == "normal": 
            sim._service_func = lambda: max(0, np.random.normal(
                loc = mean_serv_hrs,
                scale= kwargs.get("norm_scale", 0.2) * mean_serv_hrs
                ))

    ####################################################################

    def create_clinicians(sim, n_clinicians: int, config: ClinicianConfig):
        cur_in_list = len(sim.clinicians)
        for i in range(n_clinicians):
            sim.clinicians.append(
                Clinician(
                    id = i + cur_in_list,
                    appointment_time=config["appointment_length"],
                    shift_start=config["shift_pattern"][0],
                    shift_end=config["shift_pattern"][1]
                )
            )

    ####################################################################

    def get_lambda(sim):
        if sim.peak_start <= sim.clock <= sim.peak_end: #if during peak time
            if sim.peak_multiplier is not None: 
                multiplier = sim.peak_multiplier
                if sim.normal_peak: 
                    t = sim.clock 
                    centre = (sim.peak_start + sim.peak_end) /2
                    half_width = (sim.peak_end - sim.peak_start)/ 2
                    x = (t - centre) / half_width
                    k = 3 #steepness - should this be an input? 

                    shape = np.exp(-k * x ** 2)
                    edge = np.exp(-k) #normalise
                    shape = (shape - edge) / (1- edge)
                    multiplier = 1 + (sim.peak_multiplier - 1) * shape
            else:
                multiplier = np.random.choice([2,3,4]) #" the number of patient arrivals can douple, triple, or even quadruple"
            return sim.lambda_base * multiplier
        return sim.lambda_base

    def generate_interarrival(sim):
        lam = sim.get_lambda()
        sim.lambdas_t. append((lam, sim.clock))
        return np.random.exponential(1 / lam)

    def generate_service(sim):
        return sim._service_func()
    
    def get_free_clinician(sim): 
        for c in sim.clinicians: 
            if c.available and sim.clock >= c.shift_start and sim.clock <= c.shift_end:
                return c
        return None #i.e. no one is free!
    
    ####################################################################

    def arrival(sim):
        sim.num_in_system += 1
        sim.queue.append(sim.clock)
        sim.arrival_times.append(sim.clock)

        clinician = sim.get_free_clinician()

        if clinician is not None:
            arrival_time = sim.queue.popleft()

            clinician.appointment_start(sim.clock)

            service_time = sim.generate_service()
            clinician.next_available = sim.clock + service_time

            wait = sim.clock - arrival_time
            sim.waits.append(wait)

        sim.t_arrival = sim.clock + sim.generate_interarrival()

    def departure(sim, clinician: Clinician): #any mutation to clinician here will modify the original sim.clinician object 
        sim.num_in_system -= 1
        sim.departure_times.append(sim.clock)

        clinician.appointment_end(sim.clock)

        if sim.queue:
            arrival_time = sim.queue.popleft()

            clinician.appointment_start(sim.clock)

            service_time = sim.generate_service()
            clinician.next_available = sim.clock + service_time

            wait = sim.clock - arrival_time
            sim.waits.append(wait)
        else:
            clinician.next_available = None

        if sim.clock > clinician.shift_end: #enforce end of shift?
            clinician.next_available = None
            clinician.available = False

    def get_next_departure(sim): #helper function
        active = [
            (c.next_available, c)
            for c in sim.clinicians if c.next_available is not None
        ]
        return min(active, default=(float('inf'), None), key=lambda x:x[0])
        #return next availabe and clinician object (find smallest first element in list and replace with a default value of 'inf' if none)


    ####################################################################

    def step(sim): #discrete time event - we just jump to the next time where *something* happens
        assert len(sim.clinicians) >= 1, "No clinicians created. Call create_clinicians()." #kept making this mistake !!
        next_depart_time, clinician = sim.get_next_departure()

        if sim.t_arrival <= next_depart_time and sim.t_arrival <= sim.close_time: #if arrival happens next (before available server) and its before closing
            sim.clock = sim.t_arrival
            sim.arrival() #jump to arrival time and initiate arrival 
        else:
            sim.clock = next_depart_time
            if clinician is not None: 
                sim.departure(clinician)

        sim.sys_state.append((sim.clock, sim.num_in_system)) # Record system state (queue length OR total system)


*** 

## Run a simple simulation 

So that the following blocks of code make sense.
This simulation is a simple M/M/6 queue simulation. Markovian arrivals and departures (service), with 6 clinicians available for the full day, aiming for 30min appointment times, FCFS. The base arrival rate /hour is 8, ignoring effect of 'peak hours'. This simulation represents a single day of the clinic (i.e. no multisim averaging - this is not going to be used to quote metrics at this point).

In [ ]:
simple_sim_params = {
    "lambda_base" : 8,
    "appointment_time" : 30,
    "service_method" : "exponential",
    "peak_multiplier" : 1
} 

simple_sim = ClinicSim(**simple_sim_params)

simple_sim.create_clinicians( #instantiate the aforementioned clinician config
    n_clinicians= 6, 
    config= {
        "shift_pattern" : (8.0, 17.5),
        "appointment_length" : 30
    }
)

while simple_sim.clock < simple_sim.close_time: 
    simple_sim.step()


***

## Service and Arrival Rate

I modelled a few different options here and observed the resultant trends to balance what maintains the original assumptions, and what is realistic. 

We are going to plot what the serice rate (distribution of appointment times) would look like for that sim.This is using the same method as the ClinicSim instead of retrieving the small sample size of appointments from the actual sim, or separately recreating the distribution - this is helpful from a coding and data visualisation angle.
The arrival rate is constant for this sim.


In [ ]:
def plot_service_distribution_actual(sim: ClinicSim, n_samples: int = 500):

    samples = [sim.generate_service() * 60 for _ in range(n_samples)]

    fig, ax = plt.subplots(figsize=(12, 6))

    #gonna convert things from mu to times (in minutes)
    ax.hist(samples, bins=40, density=True, alpha=0.6, color="steelblue",
            edgecolor="black", label="Sampled service times")

    try: 
        sbn.kdeplot(samples, ax=ax, color="darkred", linewidth=2,
                    label="KDE (smooth density)")
    except Exception: 
        print("failed to plot seaborn kde bounds")

    ax.set_xlabel("Service time")
    ax.set_ylabel("Density")
    ax.set_title(f"Service Time Distribution. N_samples = {n_samples}")

    mean_val = np.mean(samples)
    ax.axvline(mean_val, color="green", linestyle="--", linewidth=2,
               label=f"Mean = {mean_val:.2f}")

    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_xlim(left = 0)
    ax.legend()

    fig.tight_layout()
    # print("completed plot") #For a debugging breakpoint

plot_service_distribution_actual(sim = simple_sim, n_samples = 1000)

***

## Alternative Appointment distribution modelling

I opted to instead use a different distribution for my model, as the above exponential distribution had too many very very short appointments to what I felt was realistic. I modelled a lognormal distribution and was pleased with this instead, making this, and subsequent, simulations a M/G/c queue. 

## Peak hours behaviour 

In order to emulate the behaviour of the 'peak hours' between 10.00 and 14.00hrs, I considered a random multiplier effect (coupled with a random base lambda value) but instead wanted more clarity in the affect of arrival rates on the clinic. Thus, I created an arrival rate distibution that follows a general normal bell curve within the peak window, acting as a multiplier to the base lambda. 

Note: The arrival rate plot *is* the actual service rates from the sim, so has the sampling resolution of the simulation itself, hence why not a smooth curve, but sufficiently conveys what I need it to here!

### Both choices are demonstrated in the plots below

In [ ]:

def plot_lamda_t(sim: ClinicSim): #This DOES plot the actual lambdas from the simulation!!

    lambdas, times = zip(*sim.lambdas_t)
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.step(times, lambdas, where='post', linewidth=2, label="λ(t) arrival rate")

    mean_lambda = np.mean(lambdas)
    ax.axhline(mean_lambda, color='red', linestyle='--', linewidth=1.5,
               label=f"Mean λ = {mean_lambda:.2f}")

    if hasattr(sim, "peak_start") and hasattr(sim, "peak_end"): #if provided, plot the peak hrs
        ax.axvspan(sim.peak_start, sim.peak_end, color='yellow', alpha=0.2,
                   label="Peak hours")

    ax.set_xlabel("Time of Day")
    ax.set_ylabel("Arrival Rate λ(t)")
    ax.set_title("Arrival Rate Over Time")

    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc="upper left")
    fig.tight_layout()

    print("completed plot")

#########################################################

simplev2_sim_params = {
    "lambda_base" : 8,
    "appointment_time" : 30,
    "service_method" : "lognormal",
    "peak_multiplier" : 4, # Up to 4x the baseline arrival rate at the peak time
    "peak_normal_shape": True
} 


simplev2_sim = ClinicSim(**simplev2_sim_params)

simplev2_sim.create_clinicians( #instantiate the aforementioned clinician config
    n_clinicians= 6, 
    config= {
        "shift_pattern" : (8.0, 17.5),
        "appointment_length" : 30
    }
)

while simplev2_sim.clock < simplev2_sim.close_time: 
    simplev2_sim.step()


plot_service_distribution_actual(sim = simplev2_sim, n_samples = 1000)
plot_lamda_t(sim= simplev2_sim)

***

## Finally, a sanity check plot

To show the arrivals, departures, and total patients in system throughout one simulation.

This plot clearly shows some interesting things about this system - it really struggles to recoever the backlog from the peak hours, so this is where the suggested improvements are centred around!.

In [ ]:


def plot_arrival_depart(sim: ClinicSim):

    _, ax = plt.subplots(figsize=(12,6))

    times, values = zip(*sim.sys_state)

    ax.step(times, values, where='post', label="Patients in system")

    arrival_sorted = np.sort(sim.arrival_times)
    departure_sorted = np.sort(sim.departure_times)

    ax.step(arrival_sorted, np.arange(1, len(arrival_sorted)+1),
            where='post', linestyle='--', label="Cumulative arrivals")
    ax.step(departure_sorted, np.arange(1, len(departure_sorted)+1),
            where='post', linestyle=':', label="Cumulative departures")

    ax.axvspan(sim.peak_start, sim.peak_end, alpha=0.2) # peak hours

    ax.set_xticks(np.arange(sim.open_time, sim.close_time + 0.5, 0.5)) #grid (30min intervals)
    ax.grid(True, which='both', axis='x', linestyle='--', alpha=0.5)

    ax.legend()
    ax.set_title(f"Patient arrival, departures, and total system capacity")

    # print("Finished plotting")


plot_arrival_depart(sim = simplev2_sim)

***

## Metric Calculator

Some simple numbers, including: 
* Average patient wait times
* Peak number of patients in thee queue 
* Number of patients left at the EOD
* Total number of patients served 
* Average workload of all clinicians

In [ ]:
def compute_sim_result_metrics(sim: ClinicSim) -> dict[str, float]:
    waits = np.array(sim.waits)

    peaks = [n for (_, n) in sim.sys_state]

    eod_patients = sim.num_in_system # this is EOD value by defualt if sim is complete

    patients_served = len(sim.departure_times) #every departure is a patient served

    #return a dict of each clinician and their %workload
    for clinician in sim.clinicians:
        clinician.calc_workload()
    workloads = np.array([clinician.workload for clinician in sim.clinicians])

    return {
        "avg_wait" : waits.mean(),
        "peak" : max(peaks),
        "eod_patients" : eod_patients,
        "patients_served" : patients_served,
        "avg_clinician_workload" : workloads.mean()
    }

***

## Multisims

To get some more meaningful metrics out of this, first going to make a multisim method that will come in handy a lot. This code block includes an updated simulation object which averages and aggregates all the data we get from the ClinicSim, which is handled inside the multisim method.

In [ ]:
#first we make a class that mimics ClinicSim but is more suitable for our averaged data
#should still be able to use the regular plotting methods on it tho
class AveragedSim: 
    def __init__(self, sims: list):
        self.sims= sims

        ref = sims[0]
        self.open_time = ref.open_time
        self.close_time = ref.close_time
        self.peak_start = ref.peak_start
        self.peak_end = ref.peak_end

        self.waits = np.concatenate([s.waits for s in sims])
        self.arrival_times = np.concatenate([s.arrival_times for s in sims])
        self.departure_times = np.concatenate([s.departure_times for s in sims])

        self.sys_state = self._average_series("sys_state")
        self.lambdas_t = self._average_series("lambdas_t")

    def _average_series(self, attr, dt=0.01):
        #interpolate because the time series are all different (time stamps only occuur at an event, which differs between sim)
        time_grid = np.arange(self.open_time, self.close_time, dt)
        all_interp = []

        for sim in self.sims:
            series = getattr(sim, attr, None)
            if not series:
                continue

            times, values = zip(*series)
            interp = np.interp(time_grid, times, values)
            all_interp.append(interp)

        if not all_interp:
            return []

        mean_values = np.mean(all_interp, axis=0)
        return list(zip(time_grid, mean_values))
    

def run_multisim_avg(n_sims: int, metric_sweep: tuple = None, 
                     additional_clinicians: bool = False, 
                     additional_shift_pattern: tuple[float, float] =(10.0, 14.0), 
                     appointment_length: int = 30,
                     **kwargs): 
    """
    Run n simulations and compute and return the average metrics. 
    Allows for a sweep - if sweep metric given then it runs n_sims at for each value in the sweep.

    Inputs: 
        n_sims(int) = Number of sims to run. 
        metric_sweep: tuple = ("parameter name", [values])
        **kwargs = simulation input args (for all sims). 

    Output: 
        result_dict contains:
        {
            "aggregate_sim": AveragedSim,
            "metrics": {metric: {mean, std, p95}},
            "raw_metrics": [...]
        }
    """

    def run_single_config(config_kwargs):
        sims = []
        metrics_list = []

        for i in range(n_sims):
            np.random.seed(i)

            sim = ClinicSim(**config_kwargs)
            sim.create_clinicians( #bad coding practice, but serves for this purpose
                n_clinicians= 6, 
                config= {
                    "shift_pattern" : (8.0, 18.0),
                    "appointment_length" : appointment_length
                }
            )
            if additional_clinicians: 
                sim.create_clinicians( #bit better, still not great
                    n_clinicians= 2, 
                    config= {
                        "shift_pattern" : additional_shift_pattern,
                        "appointment_length" : appointment_length
                    }
                )

            while sim.clock < sim.close_time:
                sim.step()

            sims.append(sim)
            metrics_list.append(compute_sim_result_metrics(sim))

        agg_sim = AveragedSim(sims)

        agg_metrics = {}
        keys = metrics_list[0].keys()

        for k in keys:
            vals = [m[k] for m in metrics_list]
            agg_metrics[k] = {
                "mean": np.mean(vals),
                "std": np.std(vals),
                "p95": np.percentile(vals, 95)
            }

        return {
            "aggregate_sim": agg_sim,
            "metrics": agg_metrics,
            "raw_metrics": metrics_list
        }
    
    if metric_sweep is None: 
        return run_single_config(kwargs)
    else: 
        param, values = metric_sweep
        results = {}
        for val in values:
            config_kwargs = kwargs.copy()
            config_kwargs[param] = val

            results[val] = run_single_config(config_kwargs)

        return results  

***

### Now to run a bunch of sims and understand the Clinic's performance


This is a lot of code but essentially I want to sweep base lamba and plot some metrics about the clinic. There is 4 sweeps for each of the following peak hour behaviours: 
1. No peak (flat rate lamda throughout)
2. Up to 2x peak multiplier
3. Up to 3x
4. Up to 4x



In [ ]:
def sweep_metrics_4_peaks(
    sweep_metric,
    appointment_time: float = 30,
    run_n_sims: int = 30,
    imp_1: bool = False, # Improvement for later...
    imp_shift_pattern: tuple[float, float] = (10.0, 14.0),
    base_kwargs = {
        "service_method": "lognormal",
        "peak_normal_shape": True
    }
):
    
    base_kwargs["appointment_time"] = appointment_time
    
    results = []

    for peak_multiplier in range(1, 5): # sweep
        sim_kwargs = {
            **base_kwargs,
            "peak_multiplier": peak_multiplier
        }

        result = run_multisim_avg(
            n_sims=run_n_sims,
            metric_sweep=sweep_metric,
            additional_clinicians=imp_1,
            additional_shift_pattern= imp_shift_pattern,
            **sim_kwargs
        )

        results.append(result)

    return tuple(results)


def sweep_plot(df: pd.DataFrame, 
               plot_shaded_regions: bool = True,
               y_axis_in_time: bool = True, y_axis_in_perc: bool = False):
    plt.figure(figsize=(10,6))

    # sbn.set_palette("colorblind")
    plt.rcParams.update({'font.size': 14})

    labels = df["label"].unique()
    palette = sbn.color_palette("colorblind", n_colors=len(labels))
    colour_dict = dict(zip(labels, palette))

    ax = sbn.lineplot(
        data=df,
        x="lambda_base",
        y="value",
        hue="label",
        marker="o",
        palette=colour_dict
    )
    
    annotated_values = set()# tracking to avoid duplicate labels

    # lines = plt.gca().get_lines()
    for label, subdf in df.groupby("label"):
        colour = colour_dict[label]
        if plot_shaded_regions:
            plt.fill_between(
                subdf["lambda_base"],
                subdf["value"] - subdf["std"],
                subdf["value"] + subdf["std"],
                color = colour, #american spelling :(
                alpha=0.2
            )
        left_row = subdf.iloc[0] #lowest vals
        y_val_left = left_row["value"]
        if y_axis_in_time and not y_axis_in_perc:
            annotext = format_time_hours(y_val_left)
        elif y_axis_in_perc: 
            annotext = f"{round(y_val_left *100)}%" 
        else: 
            annotext = y_val_left
        
        if annotext not in annotated_values:
            plt.text(
                left_row["lambda_base"] - 0.2,#shift slightly left
                left_row["value"],
                annotext,
                fontsize=14,
                ha="right",
                va="center"
            )
            annotated_values.add(annotext)

        max_idx = subdf["value"].idxmax()
        max_row = subdf.loc[max_idx] #highest value, dont assume its the last tho 
        y_val_max = max_row["value"]
        if y_axis_in_time and not y_axis_in_perc:
            annotext = format_time_hours(y_val_max)
        elif y_axis_in_perc:
            annotext = f"{round(y_val_max*100)}%"
        else:
            annotext = y_val_max

        if annotext not in annotated_values:
            plt.text(
                max_row["lambda_base"] + 0.2,
                y_val_max,
                annotext,
                fontsize=14,
                ha="left",
                va="center"
            )
            annotated_values.add(annotext)

    x_min, x_max = df["lambda_base"].min(), df["lambda_base"].max()
    plt.xlim(x_min - 1.5, x_max + 1.5) 
    
    ax.grid(True, which="major", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.legend(title="Peak hour behaviour scenario", loc = 'lower right')

    return ax, plt


sweep_metric = (
    "lambda_base", np.linspace(8,16,5))

results_1, results_2, results_3, results_4 = sweep_metrics_4_peaks(
    sweep_metric=sweep_metric, 
    appointment_time = 30, 
    run_n_sims=500,
    base_kwargs= simplev2_sim_params)


lamda_sweep_df = build_df_for_plot(
    results_dicts = [results_1, results_2, results_3, results_4],
    labels = ["No peak modifier", "Normal peak up to 2x", "Normal peak up to 3x", "Normal peak up to 4x"],
    sweep_metric = sweep_metric,
    metric = "avg_wait"
)

ax, plot = sweep_plot(df = lamda_sweep_df)
plot.xlabel("Base arrival rate, λ")
plot.ylabel("Average wait (hrs)")
plot.title("Average Wait Time vs Base λ for varying peak hour effects.")

workload_sweep_metrics_4_peaks_df = build_df_for_plot(
    results_dicts = [results_1, results_2, results_3, results_4],
    labels = ["No peak modifier", "Normal peak up to 2x", "Normal peak up to 3x", "Normal peak up to 4x"],
    sweep_metric = sweep_metric,
    metric = "avg_clinician_workload")

ax, plot = sweep_plot(df = workload_sweep_metrics_4_peaks_df , plot_shaded_regions=False,y_axis_in_perc = True)
vals = plt.gca().get_yticks()
plot.gca().set_yticklabels([f"{v*100:.0f}%" for v in vals])
plot.axhline(1.0, color = "black", linestyle="--", linewidth= 1.5)
plot.xlabel("Base arrival rate, λ")
plot.ylabel("Average Clinician Workload")
plot.title("Average Clinician Workload vs Base λ for varying peak hour effects.")


## Comments on plots

These plots reveal that the peak multiplier is more detrimental to the clincians workload, likely due to the surge causing a backlog which cannot be cleared, thus causing the clinicians to work 'flat out' from the surge onwards. The multiplier hasa more flat offset on customer wait times. It is interesting to see the extent of the affect on wait times from the best (lambda = 8, no peak surge) to worst (base lambda = 16, up to 4x peak multiplier) case scenarios.


We have enough data now to get some baseline metrics that are in the report 1st section:



In [ ]:

print(f"\nBase Lambda = 8, Peak multiplier = 4x")
x, y = print_sweep_results_quick(
    results = results_4,
    sweep_metric=sweep_metric,
    sim_inputs=simplev2_sim_params,
    metric = "avg_wait",
    convert_to_readable_time= True
)


print(f"\nBase Lambda = 8, Peak multiplier = 4x")
x, y = print_sweep_results_quick(
    results = results_4,
    sweep_metric=sweep_metric,
    sim_inputs=simplev2_sim_params,
    metric = "eod_patients"
)

print(f"\nBase Lambda = 8, Peak multiplier = 4x")
x, y = print_sweep_results_quick(
    results = results_4,
    sweep_metric=sweep_metric,
    sim_inputs=simplev2_sim_params,
    metric = "avg_clinician_workload"
)

print(f"\nBase Lambda = 8, Peak multiplier = 4x")
x, y = print_sweep_results_quick(
    results = results_4,
    sweep_metric=sweep_metric,
    sim_inputs=simplev2_sim_params,
    metric = "patients_served"
)

***

## Sweeping values to determine optimum application of improvements

Consider a few different shift patterns for these clinicians, to test whether it is optimal to hire them exclusively during the peak hours, or to delay to help clear the backlog. Plotting this as a heatmap with a refefrence column with no additional clinicians, and varying target appointment lengths which is the second proposed improvement.



In [ ]:

def plot_schedule_heatmap(
    results: dict[str, dict],
    baseline_results: dict[str, dict],
    appointment_lengths: list[int],
    shift_patterns: list[tuple[float, float]],
    metric: str = "avg_wait",
    calc: str = "mean",
    metric_limits: tuple[float, float] | None = None,
    title: str = ""
):
    data = []
    annot_data = []

    # build rows (appointment lengths)
    for appt in appointment_lengths:
        row = []
        annot_row = []

        base_key = f"appt{appt}_baseline" #baseline column 
        base_val = baseline_results[base_key]["metrics"][metric][calc]

        row.append(base_val)
        annot_row.append(format_time_hours(base_val))

        for shift in shift_patterns:
            key = f"appt{appt}_shift{shift[0]}-{shift[1]}"
            val = results[key]["metrics"][metric][calc]

            row.append(val)
            annot_row.append(format_time_hours(val))

        data.append(row)
        annot_data.append(annot_row)

    shift_labels = [f"Baseline"] + [f"{int(s[0])}:00-{int(s[1])}:00" for s in shift_patterns]

    df = pd.DataFrame(
        data,
        index=appointment_lengths,
        columns=shift_labels
    )

    annot_df = pd.DataFrame(
        annot_data,
        index=appointment_lengths,
        columns=shift_labels
    )

    plt.figure(figsize=(10,6))

    heatmap_kwargs = dict(
        data=df,
        cmap="RdBu_r",
        annot=annot_df,
        fmt="",
        cbar_kws={"label": "Average wait time"}
    )

    if metric_limits:
        heatmap_kwargs["vmin"] = metric_limits[0]
        heatmap_kwargs["vmax"] = metric_limits[1]

    ax = sbn.heatmap(**heatmap_kwargs)

    ax.axvline(x=1, color="white", linewidth=3)

    cbar = ax.collections[0].colorbar
    ticks = cbar.get_ticks()
    cbar.set_ticklabels([format_time_hours(t) for t in ticks])

    plt.xlabel("Shift pattern (hrs)")
    plt.ylabel("Target Appointment length (mins)")
    plt.title(title or "Clinician schedule heatmap")

    plt.tight_layout()


    plt.savefig(
        f"Heatmap_lambda{i}_highres.svg",
        bbox_inches="tight", 
        transparent=True 
    )

    plt.show()

    # print("Plotted schedule heatmap")

appointment_lengths = [20, 25, 30]
shift_patterns = [(9.0, 13.0), (10.0, 14.0), (11.0, 15.0), (12.0, 16.0)]

for i in [8,16]:

    simplev2_sim_params['lambda_base'] = i

    # Get a reference column with no additional clinicians

    baseline_results = {}

    for appt_len in appointment_lengths:
        params = {
            **simplev2_sim_params,
            "appointment_time": appt_len
        }

        key = f"appt{appt_len}_baseline"
        baseline_results[key] = run_multisim_avg(
            n_sims=100,
            additional_clinicians=False,
            appointment_length=appt_len,
            **params
        )

    # Now run each above for 3 scenarios: 10-14, 10-15, 10-16

    imp1_results = {}

    for appt_len in appointment_lengths:
        for shift in shift_patterns:
            params = {
                **simplev2_sim_params,
                "appointment_time": appt_len
            }

            key = f"appt{appt_len}_shift{shift[0]}-{shift[1]}"
            imp1_results[key] = run_multisim_avg(
                n_sims=100,
                additional_clinicians=True,
                additional_shift_pattern=shift,
                appointment_length = appt_len,
                **params
            )
        
        # print(imp1_results[key])

    x, y = print_sweep_results_quick(
        results = imp1_results,
        sweep_metric=sweep_metric,
        sim_inputs=simplev2_sim_params,
        metric = "avg_wait",
        convert_to_readable_time= True,
        baseline_results=baseline_results
    )

    x, y = print_sweep_results_quick(
        results = imp1_results,
        sweep_metric=sweep_metric,
        sim_inputs=simplev2_sim_params,
        metric = "eod_patients",
        convert_to_readable_time= False,
        baseline_results=baseline_results
    )

    x, y = print_sweep_results_quick(
        results = imp1_results,
        sweep_metric=sweep_metric,
        sim_inputs=simplev2_sim_params,
        metric = "patients_served",
        convert_to_readable_time= False,
        baseline_results=baseline_results
    )

    x, y = print_sweep_results_quick(
        results = imp1_results,
        sweep_metric=sweep_metric,
        sim_inputs=simplev2_sim_params,
        metric = "avg_clinician_workload",
        convert_to_readable_time= False,
        baseline_results=baseline_results
    )

    plot_schedule_heatmap(
        imp1_results,
        baseline_results= baseline_results,
        appointment_lengths=appointment_lengths,
        shift_patterns=shift_patterns,
        metric="avg_wait",
        # metric_limits=(0, 2),
        title=f"Average Patient Wait (Baseline arrival rate = {i}) "
    )

***

### Now to get some raw data which we once again quote in the report


In the form of % improvement from baseline


In [ ]:
imp1_results[key]